In [ ]:
# ============================================================
# Cell 1: Cài đặt & Cấu hình VnCoreNLP
# ============================================================
!pip install -q py_vncorenlp

import py_vncorenlp, os

VNCORENLP_DIR = "/content/vncorenlp"
os.makedirs(VNCORENLP_DIR, exist_ok=True)

# Tải model VnCoreNLP (chỉ cần chạy 1 lần)
py_vncorenlp.download_model(save_dir=VNCORENLP_DIR)

from py_vncorenlp import VnCoreNLP

vncorenlp_model = VnCoreNLP(
    annotators=["wseg"],
    save_dir=VNCORENLP_DIR
)

print("✅ VnCoreNLP đã sẵn sàng!")

In [ ]:
text = "Ông Nguyễn Khắc Chúc đang làm việc tại Đại học Quốc gia Hà Nội."

output = vncorenlp_model.word_segment(text)
print(output)

In [ ]:
# ============================================================
# Cell 2: Vietnamese Stopwords (auto from GitHub)
# ============================================================

import requests

STOPWORDS_URL = (
    "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/"
    "master/vietnamese-stopwords-dash.txt"
)

def get_vietnamese_stopwords() -> set[str]:
    try:
        r = requests.get(STOPWORDS_URL, timeout=10)
        r.raise_for_status()

        return {
            line.strip().lower()
            for line in r.text.splitlines()
            if line.strip() and not line.startswith("#")
        }

    except Exception as e:
        print(f"⚠️ Stopwords load failed: {e}")
        return set()


class Vietnamese:
    def __init__(self):
        self.words = get_vietnamese_stopwords()

    def exist(self, word: str) -> bool:
        """Kiểm tra từ có phải stop word không."""
        return word.lower() in self.words


vi_stopwords = Vietnamese()
print(f"✅ Vietnamese stopwords loaded: {len(vi_stopwords.words)} words")

print(list(vi_stopwords.words)[:20])

In [ ]:
# ============================================================
# Cell 3: Text — Model dữ liệu cho văn bản đã phân tích
# ============================================================

from typing import Dict, List

class Text:
    """Lưu trữ dữ liệu văn bản đã phân tích (câu, ma trận từ, dấu câu)."""

    def __init__(self):
        self._word_matrix: Dict[int, Dict[int, str]] = {}
        self._sentences: Dict[int, str] = {}
        self._marks: List[str] = []

    def set_word_matrix(self, word_matrix: Dict[int, Dict[int, str]]) -> None:
        self._word_matrix = word_matrix

    def set_sentences(self, sentences: Dict[int, str]) -> None:
        self._sentences = sentences

    def set_marks(self, marks: List[str]) -> None:
        self._marks = marks

    def get_word_matrix(self) -> Dict[int, Dict[int, str]]:
        return self._word_matrix

    def get_sentences(self) -> Dict[int, str]:
        return self._sentences

    def get_marks(self) -> List[str]:
        return self._marks

print("✅ Text model loaded")

In [ ]:
# ============================================================
# Cell 4: Parser — Phân tích văn bản (hỗ trợ VnCoreNLP)
# ============================================================
# Khi có VnCoreNLP model → dùng word_segment() để tách từ tiếng Việt
# chính xác (nhận diện từ ghép: "Hà_Nội", "trí_tuệ_nhân_tạo", …).
# Khi không có → fallback về regex split như bản gốc.

import re, string
from typing import Any, Optional

class Parser:

    def __init__(self):
        self._minimum_word_length: int = 0
        self._raw_text: str = ""
        self._marks: List[str] = []
        self._stop_words: Optional[Vietnamese] = None
        self._vncorenlp_model: Optional[Any] = None

    # ---------- setters ----------
    def set_minimum_word_length(self, word_length: int) -> None:
        self._minimum_word_length = word_length

    def set_raw_text(self, raw_text: str) -> None:
        self._raw_text = raw_text

    def set_stop_words(self, stop_words: Vietnamese) -> None:
        self._stop_words = stop_words

    def set_vncorenlp_model(self, model: Any) -> None:
        """Truyền VnCoreNLP model đã khởi tạo (annotators=['wseg'])."""
        self._vncorenlp_model = model

    def get_marks(self) -> List[str]:
        return self._marks

    # ---------- main entry ----------
    def parse(self) -> Text:
        matrix: Dict[int, Dict[int, str]] = {}
        sentences = self._get_sentences()

        for sentence_idx, sentence in enumerate(sentences):
            matrix[sentence_idx] = self._get_words(sentence)

        text = Text()
        text.set_sentences({i: s for i, s in enumerate(sentences)})
        text.set_word_matrix(matrix)
        text.set_marks(self._marks)
        return text

    # ---------- sentence splitting ----------
    def _get_sentences(self) -> List[str]:
        sentences = re.split(r'(?<=[.!?])\s+', self._raw_text.strip())
        cleaned = []
        for sentence in sentences:
            if sentence:
                cs = self._clean_sentence(sentence)
                if cs:
                    cleaned.append(cs)
        return cleaned

    # ---------- word extraction ----------
    def _get_words(self, sub_text: str) -> Dict[int, str]:
        if self._vncorenlp_model is not None:
            return self._get_words_vncorenlp(sub_text)
        return self._get_words_regex(sub_text)

    def _get_words_vncorenlp(self, sub_text: str) -> Dict[int, str]:
        """Tách từ bằng VnCoreNLP word_segment → chính xác hơn cho tiếng Việt."""
        try:
            segments = self._vncorenlp_model.word_segment(sub_text)
            # word_segment trả về list[str], mỗi phần tử là 1 câu đã tách
            raw_words = []
            for seg in segments:
                raw_words.extend(seg.split())
        except Exception:
            # Fallback nếu model lỗi
            return self._get_words_regex(sub_text)

        cleaned_words = [self._clean_word(w) for w in raw_words if w.strip()]
        cleaned_words = [w for w in cleaned_words if w]

        return self._filter_words(cleaned_words)

    def _get_words_regex(self, sub_text: str) -> Dict[int, str]:
        """Fallback: tách từ bằng regex (bản gốc dự án)."""
        words = re.split(r'(?:^\W+)|(\W*\s+\W*)|(\W+$)', sub_text)
        cleaned_words = []
        for word in words:
            if word:
                cw = self._clean_word(word)
                if cw:
                    cleaned_words.append(cw)
        return self._filter_words(cleaned_words)

    def _filter_words(self, cleaned_words: List[str]) -> Dict[int, str]:
        """Lọc từ theo stop words và độ dài tối thiểu."""
        if self._stop_words:
            filtered = [
                w for w in cleaned_words
                if not all(c in string.punctuation for c in w)
                and len(w) > self._minimum_word_length
                and not self._stop_words.exist(w)
            ]
        else:
            filtered = [
                w for w in cleaned_words
                if not all(c in string.punctuation for c in w)
                and len(w) > self._minimum_word_length
            ]
        return {i: w for i, w in enumerate(filtered)}

    # ---------- helpers ----------
    def _clean_sentence(self, sentence: str) -> str:
        trimmed = sentence.strip()
        if len(trimmed) == 1 and trimmed in string.punctuation:
            self._marks.append(trimmed)
            return ""
        return trimmed

    def _clean_word(self, word: str) -> str:
        return word.strip().lower()

print("✅ Parser loaded (VnCoreNLP-enhanced)")

In [ ]:
# ============================================================
# Cell 5: Graph — Đồ thị quan hệ giữa các từ
# ============================================================

class Graph:

    def __init__(self):
        self._graph: Dict[str, Dict[int, Dict[int, List[int]]]] = {}

    def create_graph(self, text: Text) -> None:
        """Tạo đồ thị: mỗi từ kết nối với từ liền kề trong câu."""
        word_matrix = text.get_word_matrix()

        for sentence_idx, words in word_matrix.items():
            idx_array = list(words.keys())

            for idx_key, idx_value in enumerate(idx_array):
                connections = []
                if idx_key > 0:
                    connections.append(idx_array[idx_key - 1])
                if idx_key < len(idx_array) - 1:
                    connections.append(idx_array[idx_key + 1])

                word = words[idx_value]

                if word not in self._graph:
                    self._graph[word] = {}
                if sentence_idx not in self._graph[word]:
                    self._graph[word][sentence_idx] = {}

                self._graph[word][sentence_idx][idx_value] = connections

    def get_graph(self) -> Dict[str, Dict[int, Dict[int, List[int]]]]:
        return self._graph

print("✅ Graph loaded")

In [ ]:
# ============================================================
# Cell 6: Score — Tính điểm TextRank cho từ
# ============================================================

class Score:

    def __init__(self):
        self._maximum_value: int = 0
        self._minimum_value: int = 0

    def calculate(self, graph: Graph, text: Text) -> Dict[str, float]:
        graph_data = graph.get_graph()
        word_matrix = text.get_word_matrix()

        word_connections = self._calculate_connection_numbers(graph_data)
        scores = self._calculate_scores(graph_data, word_matrix, word_connections)
        return self._normalize_and_sort_scores(scores)

    def _calculate_connection_numbers(self, graph_data) -> Dict[str, int]:
        word_connections: Dict[str, int] = {}
        for word_key, sentences in graph_data.items():
            count = 0
            for word_instances in sentences.values():
                for connections in word_instances.values():
                    count += len(connections)
            word_connections[word_key] = count
        return word_connections

    def _calculate_scores(self, graph_data, word_matrix, word_connections) -> Dict[str, int]:
        scores: Dict[str, int] = {}
        for word_key, sentences in graph_data.items():
            value = 0
            for sentence_idx, word_instances in sentences.items():
                for connections in word_instances.values():
                    for word_idx in connections:
                        if sentence_idx in word_matrix and word_idx in word_matrix[sentence_idx]:
                            word = word_matrix[sentence_idx][word_idx]
                            value += word_connections.get(word, 0)
            scores[word_key] = value
            if value > self._maximum_value:
                self._maximum_value = value
            if value < self._minimum_value or self._minimum_value == 0:
                self._minimum_value = value
        return scores

    def _normalize_and_sort_scores(self, scores: Dict[str, int]) -> Dict[str, float]:
        normalized: Dict[str, float] = {}
        for word, value in scores.items():
            normalized[word] = self._normalize(value, self._minimum_value, self._maximum_value)
        return dict(sorted(normalized.items(), key=lambda x: x[1], reverse=True))

    def _normalize(self, value: int, min_val: int, max_val: int) -> float:
        divisor = max_val - min_val
        return (value - min_val) / divisor if divisor != 0 else 0.0

print("✅ Score loaded")

In [ ]:
# ============================================================
# Cell 7: Summarize — Chọn câu quan trọng
# ============================================================

class Summarize:
    GET_ALL_IMPORTANT = 0
    GET_FIRST_IMPORTANT_AND_FOLLOWINGS = 1

    def __init__(self):
        self._sentence_weight: Dict[int, int] = {}

    def get_summarize(
        self,
        scores: Dict[str, float],
        graph: Graph,
        text: Text,
        key_word_limit: int,
        sentence_limit: int,
        summarize_type: int,
    ) -> List[str]:
        graph_data = graph.get_graph()
        sentences = text.get_sentences()
        marks = text.get_marks()

        self._find_and_weight_sentences(scores, graph_data, key_word_limit)

        if summarize_type == self.GET_ALL_IMPORTANT:
            return self._get_all_important(sentences, marks, sentence_limit)
        elif summarize_type == self.GET_FIRST_IMPORTANT_AND_FOLLOWINGS:
            return self._get_first_important_and_followings(sentences, marks, sentence_limit)
        return []

    def _find_and_weight_sentences(self, scores, graph_data, key_word_limit) -> None:
        i = 0
        for word, score in scores.items():
            if i >= key_word_limit:
                break
            i += 1
            if word in graph_data:
                for sentence_idx in graph_data[word].keys():
                    self._update_sentence_weight(sentence_idx)
        self._sentence_weight = dict(
            sorted(self._sentence_weight.items(), key=lambda x: x[1], reverse=True)
        )

    def _get_all_important(self, sentences, marks, sentence_limit) -> List[str]:
        summary: Dict[int, str] = {}
        i = 0
        for sentence_idx, weight in self._sentence_weight.items():
            if i >= sentence_limit:
                break
            i += 1
            summary[sentence_idx] = sentences[sentence_idx] + self._get_mark(marks, sentence_idx)
        return list(dict(sorted(summary.items())).values())

    def _get_first_important_and_followings(self, sentences, marks, sentence_limit) -> List[str]:
        summary: Dict[int, str] = {}
        start_idx = 0
        for sentence_idx, weight in self._sentence_weight.items():
            summary[sentence_idx] = sentences[sentence_idx] + self._get_mark(marks, sentence_idx)
            start_idx = sentence_idx
            break
        i = 0
        for sentence_idx, sentence in sentences.items():
            if sentence_idx <= start_idx:
                continue
            elif i >= sentence_limit - 1:
                break
            i += 1
            summary[sentence_idx] = sentences[sentence_idx] + self._get_mark(marks, sentence_idx)
        return list(summary.values())

    def _update_sentence_weight(self, sentence_idx: int) -> None:
        if sentence_idx in self._sentence_weight:
            self._sentence_weight[sentence_idx] += 1
        else:
            self._sentence_weight[sentence_idx] = 1

    def _get_mark(self, marks: List[str], idx: int) -> str:
        return marks[idx] if idx < len(marks) else ""

print("✅ Summarize loaded")

In [ ]:
# ============================================================
# Cell 8: TextRankFacade — API chính (VnCoreNLP-enhanced)
# ============================================================
import math
from typing import Any

class TextRankFacade:
    """Facade cho thuật toán TextRank tóm tắt văn bản tiếng Việt.

    So với bản gốc:
    - Thêm set_vncorenlp_model() để Parser dùng VnCoreNLP tách từ.
    - Thêm summarize() với logic tự động chọn số câu:
        • ≤ 5 câu  → chọn tối đa 3 câu quan trọng nhất
        • > 5 câu  → chọn ~40% số câu, tối thiểu 5 câu
    """

    GET_ALL_IMPORTANT = Summarize.GET_ALL_IMPORTANT
    GET_FIRST_IMPORTANT_AND_FOLLOWINGS = Summarize.GET_FIRST_IMPORTANT_AND_FOLLOWINGS

    def __init__(self):
        self._stop_words: Optional[Vietnamese] = None
        self._vncorenlp_model: Optional[Any] = None

    # ---------- config ----------
    def set_stop_words(self, stop_words: Vietnamese) -> None:
        self._stop_words = stop_words

    def set_vncorenlp_model(self, model: Any) -> None:
        """Truyền VnCoreNLP model (annotators=['wseg']) để cải thiện word segmentation."""
        self._vncorenlp_model = model

    # ---------- internal: tạo parser đã config ----------
    def _create_parser(self, raw_text: str) -> Parser:
        parser = Parser()
        parser.set_minimum_word_length(3)
        parser.set_raw_text(raw_text)
        if self._stop_words:
            parser.set_stop_words(self._stop_words)
        if self._vncorenlp_model:
            parser.set_vncorenlp_model(self._vncorenlp_model)
        return parser

    # ---------- public API chính ----------
    def summarize(self, raw_text: str) -> List[str]:
        """Tóm tắt văn bản với số câu tự động:
        - ≤ 5 câu  → chọn tối đa 3 câu quan trọng nhất
        - > 5 câu  → chọn ~40% số câu, tối thiểu 5 câu
        """
        text = self._create_parser(raw_text).parse()
        sentences = text.get_sentences()
        n = len(sentences)

        # Tính số câu tóm tắt
        if n <= 5:
            k = min(n, 3)
        else:
            k = max(5, math.ceil(n * 0.4))

        # Số keyword phân tích: scale theo độ dài văn bản
        analyzed_keywords = min(max(5, n), 15)

        graph = Graph()
        graph.create_graph(text)
        scores = Score().calculate(graph, text)

        return Summarize().get_summarize(
            scores, graph, text,
            analyzed_keywords, k,
            Summarize.GET_ALL_IMPORTANT,
        )

    # ---------- các method khác (giữ tương thích) ----------
    def get_only_keywords(self, raw_text: str) -> Dict[str, float]:
        """Trích xuất từ khóa và điểm số."""
        text = self._create_parser(raw_text).parse()
        graph = Graph()
        graph.create_graph(text)
        return Score().calculate(graph, text)

    def get_highlights(self, raw_text: str) -> List[str]:
        """Lấy các câu nổi bật (15–25% số câu, min 2, max 6)."""
        text = self._create_parser(raw_text).parse()
        sentences = text.get_sentences()
        n_sent = len(sentences)
        maximum_sentences = min(max(2, math.ceil(n_sent * 0.2)), 6)
        analyzed_keywords = min(max(5, n_sent), 12)

        graph = Graph()
        graph.create_graph(text)
        scores = Score().calculate(graph, text)

        return Summarize().get_summarize(
            scores, graph, text,
            analyzed_keywords, maximum_sentences,
            Summarize.GET_ALL_IMPORTANT,
        )

    def summarize_text_freely(
        self, raw_text: str,
        analyzed_keywords: int,
        expected_sentences: int,
        summarize_type: int,
    ) -> List[str]:
        """Tùy chỉnh tóm tắt theo tham số."""
        text = self._create_parser(raw_text).parse()
        graph = Graph()
        graph.create_graph(text)
        scores = Score().calculate(graph, text)
        return Summarize().get_summarize(
            scores, graph, text,
            analyzed_keywords, expected_sentences,
            summarize_type,
        )

print("✅ TextRankFacade loaded (VnCoreNLP-enhanced)")

In [ ]:
# ============================================================
# Cell 9: Văn bản mẫu
# ============================================================

sample_text = """
Trong kỳ điều chỉnh chiều ngày 5/3, giá xăng dầu đồng loạt tăng mạnh. Trong đó, giá xăng tăng 1.900-2.190 đồng/lít, giá dầu tăng 1.800-7.140 đồng/lít.
Chiều 5/3, liên Bộ Công Thương - Tài chính điều chỉnh giá bán lẻ xăng dầu, áp dụng từ 15h cùng ngày.

Cơ quan điều hành quyết định tăng 1.920 đồng/lít với xăng E5 RON 92 và tăng 2.190 đồng/lít với xăng RON 95. Sau điều chỉnh, giá bán lẻ với xăng E5 RON 92 là 21.440 đồng/lít và xăng RON 95 là 22.340 đồng/lít.

Trong khi đó, dầu diesel tăng 3.760 đồng/lít lên 23.030 đồng/lít, dầu hỏa tăng 7.140 đồng/lít lên 26.600 đồng/lít; dầu mazut tăng 1.810 đồng/lít lên 17.490 đồng/kg. Cơ quan điều hành vẫn duy trì không trích hay chi quỹ bình ổn giá.

Như vậy, giá xăng trong nước đã tăng mạnh 2 phiên liên tiếp chỉ sau một phiên giảm. Từ đầu năm đến nay, giá xăng có 6 lần tăng, 4 lần giảm.

Về số dư Quỹ bình ổn giá xăng dầu, đến hết quý III/2025, tổng số dư Quỹ bình ổn giá xăng dầu của 27 thương nhân đầu mối là 5.617 tỷ đồng, tăng gần 3 tỷ đồng so với quý trước đó.

Chiến sự giữa Mỹ - Israel và Iran đang tạo ra những xáo trộn sâu rộng đối với dòng chảy dầu khí toàn cầu, qua đó tác động trực tiếp đến các nền kinh tế phụ thuộc nhập khẩu năng lượng như Việt Nam. Về nguồn cung xăng dầu, nhiều doanh nghiệp đầu mối cho biết nguồn cung xăng dầu vẫn được đảm bảo.

Đại diện Tập đoàn Xăng dầu Việt Nam (Petrolimex) cho biết lượng tồn kho toàn hệ thống đầu tháng 3 được duy trì theo quy định Nghị định kinh doanh xăng dầu, đủ đáp ứng nhu cầu phân phối.

Doanh nghiệp cũng chủ động kế hoạch tạo nguồn năm 2026, ký hợp đồng với hai nhà máy lọc dầu trong nước và bố trí nhập khẩu theo kế hoạch. Riêng 10 ngày đầu tháng 3, sản lượng xăng dầu nhập khẩu tăng khoảng 50% so với mức bình quân tháng.

Tương tự, Tập đoàn Công nghiệp - Năng lượng Quốc gia Việt Nam (Petrovietnam) cho biết đã yêu cầu các đơn vị tăng nguồn dự phòng, duy trì công suất nhà máy và chuẩn bị phương án nhập khẩu bổ sung. Nguồn cung xăng dầu trong nước được khẳng định vẫn đảm bảo trong vài tháng tới.

Tại Công ty cổ phần Lọc hóa dầu Bình Sơn (BSR), đơn vị quản lý Nhà máy Lọc dầu Dung Quất, khoảng 30-35% dầu thô đầu vào hiện được nhập khẩu từ nhiều khu vực. Doanh nghiệp đã ký hợp đồng mua khoảng 3 triệu thùng dầu thô giai đoạn tháng 3-5 để duy trì vận hành nhà máy ở mức 118-120% công suất thiết kế.'
"""

print(f"Văn bản gốc: {len(sample_text.strip().splitlines())} câu")

In [ ]:
# ============================================================
# Cell 10: Chạy tóm tắt — VỚI VnCoreNLP
# ============================================================
# Logic số câu:
#   ≤ 5 câu  → chọn tối đa 3 câu quan trọng nhất
#   > 5 câu  → chọn ~40% số câu, tối thiểu 5 câu

tr = TextRankFacade()
tr.set_stop_words(Vietnamese())
tr.set_vncorenlp_model(vncorenlp_model)  # ← model từ Cell 1

# Đếm số câu gốc
n_sentences = len(sample_text.strip().splitlines())
if n_sentences <= 5:
    expected = min(n_sentences, 3)
else:
    expected = max(5, math.ceil(n_sentences * 0.4))

print(f"� Văn bản gốc: {n_sentences} câu → tóm tắt: {expected} câu")
print("=" * 60)

summary = tr.summarize(sample_text)

print("\n📝 BẢN TÓM TẮT:\n")
for i, s in enumerate(summary, 1):
    print(f"  {i}. {s}")

print(f"\n✅ Đã chọn {len(summary)}/{n_sentences} câu quan trọng nhất.")